# Descripteurs Python : valide tous tes attributs avec une seule classe

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moncoachdata/MonShotData/blob/main/Python/Descripteurs-Python.ipynb)

📖 **Le shot complet** : [Descripteurs Python : valide tous tes attributs en une seule classe](https://www.monshotdata.com/p/python-descripteurs-valider-attributs-poo)

---

Avec `@property`, il faut écrire un getter et un setter pour **chaque** attribut à valider, et la validation saute à la création de l'objet. Les **descripteurs** règlent les deux problèmes : tu écris la règle de validation une seule fois, dans une classe, et tu l'attaches à autant d'attributs que tu veux.

**À la fin de ce notebook, tu sauras :**

- pourquoi `@property` devient répétitif dès que plusieurs attributs partagent la même règle, et pourquoi il laisse passer une valeur invalide à la création de l'objet ;
- pourquoi une fonction de validation appelée partout ne règle qu'une partie du problème ;
- comment écrire un descripteur avec `__set_name__`, `__set__` et `__get__`, et le réutiliser sur plusieurs attributs.

### Avant de lancer

- **Installation** : rien : ce notebook n'utilise que Python standard, sans bibliothèque ni données.
- **Les erreurs voulues** : plusieurs cellules provoquent exactement l'erreur dont parle le shot. Elle est capturée par un `try / except` et affichée, pour que « Tout exécuter » aille au bout sans s'arrêter.

🧪 *Testé le 22 novembre 2026 : exécution complète de haut en bas (Python 3.13). Les sorties affichées viennent de cette exécution.*

## 1. Le décorateur `@property` : un getter et un setter par attribut

On veut une classe dont les trois attributs doivent rester **positifs**. Avec `@property`, chaque attribut a son propre getter (qui renvoie la valeur) et son propre setter (qui la valide) : six méthodes quasi identiques.

In [1]:
class DummyClass:

    def __init__(self, num1, num2, num3):
        self._number1 = num1
        self._number2 = num2
        self._number3 = num3

    @property
    def number1(self):
        return self._number1

    @number1.setter
    def number1(self, value):
        if value < 0:
            raise ValueError("Enter positive value")
        self._number1 = value

    @property
    def number2(self):
        return self._number2

    @number2.setter
    def number2(self, value):
        if value < 0:
            raise ValueError("Enter positive value")
        self._number2 = value

    @property
    def number3(self):
        return self._number3

    @number3.setter
    def number3(self, value):
        if value < 0:
            raise ValueError("Enter positive value")
        self._number3 = value

Les trois setters répètent le même `if`. Mais le vrai problème est ailleurs : regarde ce qui se passe à la **création** de l'objet.

In [2]:
a = DummyClass(-1, -2, -3)   # trois valeurs négatives, et Python ne lève aucune erreur
print(a.number1, a.number2, a.number3)

-1 -2 -3


`__init__` écrit directement dans `_number1`, `_number2` et `_number3` : il **contourne les setters**, donc la validation ne s'exécute jamais. Elle ne se déclenche que si on réassigne l'attribut ensuite :

In [3]:
try:
    a.number1 = -5
except ValueError as e:
    print("ValueError :", e)

ValueError : Enter positive value


## 2. Une solution bancale : la fonction de validation

Pour éviter la redondance, on sort la règle dans une fonction `checkPositive`, puis on l'appelle partout où c'est nécessaire, y compris dans `__init__`.

In [4]:
def checkPositive(value):
    if value < 0:
        raise ValueError("Enter positive value")

In [5]:
class DummyClass:

    def __init__(self, num1, num2, num3):
        checkPositive(num1)
        checkPositive(num2)
        checkPositive(num3)
        self._number1 = num1
        self._number2 = num2
        self._number3 = num3

    @property
    def number1(self):
        return self._number1

    @number1.setter
    def number1(self, value):
        checkPositive(value)
        self._number1 = value

    @property
    def number2(self):
        return self._number2

    @number2.setter
    def number2(self, value):
        checkPositive(value)
        self._number2 = value

    @property
    def number3(self):
        return self._number3

    @number3.setter
    def number3(self, value):
        checkPositive(value)
        self._number3 = value

In [6]:
try:
    DummyClass(-1, 2, 3)   # cette fois, l'erreur est bien levée
except ValueError as e:
    print("ValueError :", e)

ValueError : Enter positive value


Ça marche, mais le code reste lourd :

- on a toujours des appels de fonction explicites et redondants dans chaque setter ;
- les getters font tous la même chose (renvoyer l'attribut) ;
- `__init__` est maintenant pollué par les appels de validation.

## 3. Les descripteurs

Un **descripteur** est un objet doté de méthodes spéciales (`__get__`, `__set__`...) qui gère l'accès à **un** attribut d'une autre classe. Ici, chaque attribut de `DummyClass` aura son propre objet descripteur, et tous seront créés à partir de la même classe.

### 3.1 La classe descripteur

Une classe descripteur typique s'écrit avec trois méthodes :

| Méthode | Quand est-elle appelée ? | Rôle |
|---|---|---|
| `__set_name__(self, owner, name)` | quand le descripteur est assigné à un attribut de classe | mémoriser le **nom** de l'attribut auquel il est rattaché |
| `__set__(self, instance, value)` | quand on **assigne** une valeur à l'attribut | contrôler la valeur, puis la stocker |
| `__get__(self, instance, owner)` | quand on **lit** l'attribut | renvoyer la valeur stockée |

Les paramètres : `instance` est l'objet (un `DummyClass()`), `owner` est la classe elle-même (`DummyClass`), `value` est la valeur assignée et `name` le nom de l'attribut.

In [7]:
class DescriptorClass:

    def __set_name__(self, owner, name):
        self.attribute_name = name

    def __set__(self, instance, value):
        if value < 0:
            raise ValueError("Enter positive value")
        instance.__dict__[self.attribute_name] = value

    def __get__(self, instance, owner):
        if instance is None:        # accès depuis la classe (DummyClass.number) : on renvoie le descripteur
            return self
        return instance.__dict__[self.attribute_name]

Le stockage se fait dans `instance.__dict__` (le dictionnaire de **l'objet**), pas dans le descripteur lui-même. C'est important : le descripteur est créé une seule fois au niveau de la **classe**, donc partagé par tous les objets. S'il stockait la valeur dans `self`, tous les objets verraient la même valeur.

### 3.2 Un attribut

La ligne `number = DescriptorClass()` crée le descripteur **au niveau de la classe**. `__init__` assigne ensuite `self.number = num`, ce qui déclenche automatiquement `__set__`.

In [8]:
class DummyClass:
    number = DescriptorClass()

    def __init__(self, num):
        self.number = num

In [9]:
try:
    DummyClass(-1)   # la validation s'exécute dès la création, sans aucun contrôle dans __init__
except ValueError as e:
    print("ValueError :", e)

ValueError : Enter positive value


In [10]:
dummyObj = DummyClass(1)
try:
    dummyObj.number = -1   # et à chaque nouvelle assignation
except ValueError as e:
    print("ValueError :", e)

ValueError : Enter positive value


### 3.3 Plusieurs attributs

Pour valider trois attributs, on crée trois descripteurs : **un seul `DescriptorClass`, plus de getters ni de setters**.

In [11]:
class DummyClass:

    number1 = DescriptorClass()
    number2 = DescriptorClass()
    number3 = DescriptorClass()

    def __init__(self, num1, num2, num3):
        self.number1 = num1
        self.number2 = num2
        self.number3 = num3

In [12]:
try:
    DummyClass(-1, -2, -3)   # entrée invalide : erreur dès la création
except ValueError as e:
    print("ValueError :", e)

ValueError : Enter positive value


In [13]:
dummyObj = DummyClass(1, 2, 3)   # entrée valide
print(dummyObj.number1, dummyObj.number2, dummyObj.number3)

1 2 3


In [14]:
for attribut, valeur in [("number1", -1), ("number2", -2), ("number3", -3)]:
    try:
        setattr(dummyObj, attribut, valeur)   # équivalent de dummyObj.number1 = -1, etc.
    except ValueError as e:
        print(f"{attribut} = {valeur} -> ValueError : {e}")

number1 = -1 -> ValueError : Enter positive value
number2 = -2 -> ValueError : Enter positive value
number3 = -3 -> ValueError : Enter positive value


### 3.4 Chaque objet garde ses propres valeurs

Vérifions que le descripteur, partagé par la classe, ne mélange pas les valeurs des objets :

In [15]:
a = DummyClass(1, 2, 3)
b = DummyClass(10, 20, 30)

print("a.number1 =", a.number1, "| b.number1 =", b.number1)
print("Valeurs stockées dans a :", a.__dict__)
print("Valeurs stockées dans b :", b.__dict__)

a.number1 = 1 | b.number1 = 10
Valeurs stockées dans a : {'number1': 1, 'number2': 2, 'number3': 3}
Valeurs stockées dans b : {'number1': 10, 'number2': 20, 'number3': 30}


Les valeurs vivent bien dans le dictionnaire de chaque objet. Et si on lit l'attribut **depuis la classe**, c'est le descripteur lui-même qu'on obtient (grâce au `if instance is None` écrit plus haut) :

In [16]:
print(type(DummyClass.number1).__name__)

DescriptorClass


## 4. Ce qu'il faut retenir

- `@property` oblige à écrire un getter et un setter par attribut, et la validation ne se déclenche pas dans `__init__` si celui-ci écrit directement l'attribut « privé ».
- Un **descripteur** regroupe la règle de validation dans une seule classe, réutilisable sur autant d'attributs que nécessaire.
- `__set_name__` mémorise le nom de l'attribut, `__set__` valide puis stocke, `__get__` renvoie la valeur stockée.
- Le descripteur vit au niveau de la **classe** : stocke les valeurs dans `instance.__dict__`, jamais dans le descripteur, sinon tous les objets partageraient la même valeur.
- Dans `__get__`, gère le cas `instance is None` (accès depuis la classe) en renvoyant le descripteur.

---

## Pour aller plus loin

- [Python ne respecte pas vraiment l'encapsulation en POO](https://www.monshotdata.com/p/python-encapsulation-poo) — pourquoi `_attribut` n'est qu'une convention
- [Booste la notation pointée avec les getters & setters !](https://www.monshotdata.com/p/notation-point-getters-setters) — le décorateur `@property` en détail
- [Programmation orientée objet avec Python pour les data scientists](https://www.monshotdata.com/p/poo-python-data-scientists-partie1) — l'analyse complète de la POO avancée
- [Tous les notebooks des shots](https://github.com/moncoachdata/MonShotData) — le code Python de la newsletter, rangé par thème